# Chunk Evaluation Pipeline
Build chunk variants, run benchmark, evaluate, and compare results.

In [1]:
from pathlib import Path
import subprocess, shutil

def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for cur in [p] + list(p.parents):
        if (cur / 'config.py').exists() and (cur / 'data').exists():
            return cur
    raise RuntimeError('Cannot find repo root')

ROOT = find_repo_root(Path.cwd())
print('ROOT =', ROOT)

def run(cmd):
    print('\n$ ' + ' '.join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)

GT = ROOT / 'data' / 'qa_pairs' / 'eval_ground_truth.jsonl'
if not GT.exists():
    raise FileNotFoundError(f'Ground truth not found: {GT}')

ROOT = /Users/ben/Library/CloudStorage/GoogleDrive-rs.runsheng@gmail.com/My Drive/Colab Notebooks/CS614 GenAI/qa_agent_legal_tax


FileNotFoundError: Ground truth not found: /Users/ben/Library/CloudStorage/GoogleDrive-rs.runsheng@gmail.com/My Drive/Colab Notebooks/CS614 GenAI/qa_agent_legal_tax/data/qa_pairs/eval_ground_truth.jsonl

In [ ]:
# 1) Build two chunk variants
run(['python', '-m', 'src.eval.chunking'])

In [ ]:
# 2) Swap active chunk dir and run eval for each strategy
active = ROOT / 'data' / 'acts_chunked'
fixed = ROOT / 'data' / 'acts_chunked_fixed'
struct = ROOT / 'data' / 'acts_chunked_struct'
backup = ROOT / 'data' / 'acts_chunked__backup_for_chunk_eval'

def activate_variant(src_dir: Path):
    if backup.exists():
        shutil.rmtree(backup)
    if active.exists():
        active.rename(backup)
    shutil.copytree(src_dir, active)

def restore_active():
    if active.exists():
        shutil.rmtree(active)
    if backup.exists():
        backup.rename(active)

try:
    # fixed
    activate_variant(fixed)
    run(['python', 'scripts/eval/run_eval_benchmark.py', '--gt', str(GT), '--out', 'outputs/preds_fixed.jsonl', '--enable-rerank', 'false'])
    run(['python', 'scripts/eval/evaluate_predictions.py', '--gt', str(GT), '--pred', 'outputs/preds_fixed.jsonl', '--k', '5', '--out', 'outputs/eval_fixed.json'])

    # struct
    activate_variant(struct)
    run(['python', 'scripts/eval/run_eval_benchmark.py', '--gt', str(GT), '--out', 'outputs/preds_struct.jsonl', '--enable-rerank', 'false'])
    run(['python', 'scripts/eval/evaluate_predictions.py', '--gt', str(GT), '--pred', 'outputs/preds_struct.jsonl', '--k', '5', '--out', 'outputs/eval_struct.json'])
finally:
    restore_active()

In [ ]:
# 3) Compare and print markdown result
run([
    'python', 'scripts/eval/chunk_eval/eval_chunking_compare.py',
    '--fixed', 'outputs/eval_fixed.json',
    '--struct', 'outputs/eval_struct.json',
    '--fixed-name', 'Fixed-size chunking',
    '--struct-name', 'Structure-aware chunking',
    '--out-md', 'outputs/eval_chunking_comparison.md',
    '--out-json', 'outputs/eval_chunking_comparison.json'
])

print('\n===== outputs/eval_chunking_comparison.md =====')
print((ROOT / 'outputs' / 'eval_chunking_comparison.md').read_text(encoding='utf-8'))